[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-recommender.ipynb)

# Recommender Systems

*AIBits Academy · Machine Learning End To End · Unsupervised / Hybrid Learning · New*

The algorithm family behind "customers who bought this also bought," Flipkart's homepage, and Swiggy's restaurant ranking — an explicit component of Andrew Ng's Machine Learning Specialization.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Two Fundamental Approaches

|  | Content-Based Filtering | Collaborative Filtering |
|---|---|---|
| Uses | Item features (genre, price, brand, description) | User-item interaction patterns only (ratings, clicks, purchases) |
| Needs item metadata? | Yes — rich features required | No — works purely from behaviour |
| New-item problem | Handles new items fine (uses their features) | Cold-start — no interactions yet to learn from |
| Serendipity | Low — recommends more of the same | Higher — can surface unexpected items liked by similar users |

## Collaborative Filtering — The User-Item Matrix

Represent all interactions as a sparse matrix R, where Rᵤᵢ is user u's rating of item i (mostly empty — no user rates more than a tiny fraction of a catalogue):

In [ ]:
import pandas as pd
import numpy as np

# Zomato-style user-restaurant rating matrix (1-5, NaN = not yet rated)
ratings = pd.DataFrame({
    'Punjabi Tadka':   [5, np.nan, 4, 1, np.nan],
    'Sagar Ratna':     [4, 5, np.nan, 2, 3],
    'Barbeque Nation': [np.nan, 4, 5, np.nan, 4],
    'Cafe Coffee Day': [2, 1, np.nan, 5, 4],
}, index=['Priya','Rahul','Anita','Vikram','Sara'])
print(ratings)

## Matrix Factorization — SVD-Based Collaborative Filtering

The core idea: assume the true (fully-observed) rating matrix has low **latent rank** — every user and every item can be described by a small number of hidden "taste dimensions" (spice tolerance, price sensitivity, cuisine preference), and a rating is just the dot product of a user's taste vector and an item's taste vector:

$$R \approx \mathbf{U} \cdot \mathbf{V}^\mathsf{T} \qquad \hat{R}_{ui} = \mathbf{u}_u \cdot \mathbf{v}_i \qquad \text{where } \mathbf{U} \in \mathbb{R}^{\text{users}\times k},\ \mathbf{V} \in \mathbb{R}^{\text{items}\times k}$$

This is learned by minimising squared reconstruction error over only the *observed* entries (plus regularisation to avoid overfitting the sparse observations):

$$\min_{\mathbf{U},\mathbf{V}} \sum_{(u,i)\,\text{observed}} \left(R_{ui} - \mathbf{u}_u\cdot\mathbf{v}_i\right)^2 \;+\; \lambda\left(\lVert \mathbf{U} \rVert^2 + \lVert \mathbf{V} \rVert^2\right)$$

In [ ]:
from sklearn.decomposition import NMF

# Fill missing with 0 for this simple demo (production systems mask observed vs missing explicitly)
R = ratings.fillna(0).values

# Non-negative Matrix Factorization — decompose into 2 latent taste dimensions
model = NMF(n_components=2, init='random', random_state=42, max_iter=500)
U = model.fit_transform(R)   # user latent factors (5 users × 2 dims)
V = model.components_        # item latent factors (2 dims × 4 items)

R_hat = U @ V
predicted = pd.DataFrame(R_hat, index=ratings.index, columns=ratings.columns).round(2)
# Anita never rated "Cafe Coffee Day" — this is the model's predicted rating
print(f"Predicted rating, Anita → Cafe Coffee Day: {predicted.loc['Anita','Cafe Coffee Day']}")
# Recommend the highest-predicted UNRATED item for each user
for user in ratings.index:
    unrated = ratings.loc[user][ratings.loc[user].isna()].index
    if len(unrated):
        best = predicted.loc[user, unrated].idxmax()
        print(f"  Recommend to {user}: {best} (predicted {predicted.loc[user,best]:.2f})")

## ⚠ Advanced: Factorization Machines & Field-Aware Factorization Machines

The Matrix Factorization above only uses two signals: which user, which item. Real production systems have far more — time of day, device type, promotion applied, restaurant category. **Factorization Machines (FM)** generalise matrix factorization to handle an arbitrary feature vector, learning a latent vector **v**ᵢ for *every* feature (not just user-ID and item-ID) and modelling all pairwise feature interactions efficiently through those vectors, even for features that never co-occur often enough to estimate a direct interaction weight from data alone:

$$\hat{y}(x) = w_0 + \sum_i w_i x_i + \sum_i \sum_{j>i} \langle \mathbf{v}_i, \mathbf{v}_j \rangle x_i x_j$$

**Field-Aware Factorization Machines (FFM)** — the model behind several winning solutions on Kaggle's Criteo/Avazu click-through-rate competitions — refine this further: features are grouped into *fields* (e.g. all restaurant-category dummy variables form one field, all city dummy variables form another), and each feature learns a *separate* latent vector for every field it might interact with, rather than one single vector used everywhere:

$$\hat{y}(x) = w_0 + \sum_i w_i x_i + \sum_i \sum_{j>i} \langle \mathbf{v}_{i,f(j)}, \mathbf{v}_{j,f(i)} \rangle x_i x_j \qquad \text{where } f(i) = \text{the field feature } i \text{ belongs to}$$

Intuitively: how "spicy cuisine" interacts with "Bengaluru" and how "spicy cuisine" interacts with "evening order" may be genuinely different relationships — FFM lets the model represent that, at the cost of more parameters and a real risk of overfitting on smaller datasets, which is why FFM tends to win specifically on the very large, feature-rich, click-through-rate-style datasets it was designed for rather than smaller recommendation problems where plain FM or ordinary Matrix Factorization already suffice.

## Neighborhood-Based Collaborative Filtering — Similarity Metrics

Before matrix factorization became dominant, the classic collaborative-filtering approach was **memory-based**: never learn latent factors at all — just find similar users (or similar items) directly from the rating matrix, and predict from their actual ratings.

|  | User-Based CF | Item-Based CF |
|---|---|---|
| Idea | Find users similar to Priya (by rating pattern); predict from what they rated | Find items similar to ones Priya already rated (by who else rated them); predict from her own ratings of those |
| Similarity computed between | Rows of the rating matrix (users) | Columns of the rating matrix (items) |
| Stability | Recomputes as user base grows (unstable at scale) | Item catalogues change slower than user bases — more stable in production |

$$\hat{y}(u,i) = \dfrac{\sum_v \mathrm{sim}(u,v)\cdot r(v,i)}{\sum_v |\mathrm{sim}(u,v)|} \qquad \text{(weighted average of similar users } v\text{'s ratings for item } i\text{)}$$

The choice of similarity metric matters a great deal — it's the single biggest lever in a neighborhood-based system:

| Metric | What it does | Best for |
|---|---|---|
| Cosine Similarity | Angle between two rating vectors, ignoring magnitude | Simple baseline; already used above for content features |
| Adjusted Cosine | Cosine similarity after subtracting each user's own mean rating first | Item-based CF — corrects for users who rate everything generously or harshly |
| Pearson Correlation | Cosine similarity computed on mean-centred vectors — a standard linear correlation | User-based CF — same rater-bias correction, from the user's side |
| Spearman Rank Correlation | Pearson computed on the *ranks* of ratings, not the raw values | When rating scales are inconsistent or only ordinal (1-5 stars meaning different things to different raters) |
| Mean Squared Difference (MSD) | Inverse of the average squared difference in ratings on commonly-rated items | Fast, simple; sensitive to the absolute rating scale |
| Jaccard Similarity | \|items both interacted with\| / \|items either interacted with\| — ignores rating value entirely | Implicit feedback (clicks, views, purchases) where there's no numeric rating, only yes/no |

In [ ]:
from scipy.stats import pearsonr
import itertools

# A denser 8-user sample of the same Zomato-style ratings — item-based CF needs enough
# overlapping raters per pair to compute a meaningful correlation, unlike the earlier 5-user matrix
ratings2 = pd.DataFrame({
    'Punjabi Tadka':   [5,4,4,1,2,5,np.nan,3],
    'Sagar Ratna':     [4,5,3,2,1,4,5,2],
    'Barbeque Nation': [2,3,5,4,5,2,4,np.nan],
    'Cafe Coffee Day': [1,2,4,5,4,1,3,5],
}, index=['Priya','Rahul','Anita','Vikram','Sara','Kabir','Meera','Arjun'])

def pearson_sim(item_a, item_b):
    both_rated = ratings2[[item_a, item_b]].dropna()
    r, _ = pearsonr(both_rated[item_a], both_rated[item_b])
    return r

for a, b in itertools.combinations(ratings2.columns, 2):
    print(f"{a} <-> {b}: Pearson r = {pearson_sim(a,b):.2f}")

The pattern makes intuitive sense: Punjabi Tadka and Sagar Ratna (both North Indian, spicier) correlate positively — users who like one tend to like the other — while Barbeque Nation and Cafe Coffee Day (different cuisine/price bracket entirely) also correlate positively with each other but negatively with the North Indian pair. At production scale (millions of ratings) these correlations, computed entirely from behaviour with no item metadata at all, become the backbone of "people who rated X highly also rated Y highly" recommendations.

## Try It — Watch Item-Based Similarity Shift Live

This is the exact 8-user ratings matrix from above, made editable. Pick any two restaurants, then drag any user's rating slider for either one — the Pearson correlation and the scatter of (rating A, rating B) pairs recompute live, using the same pairwise-deletion logic as `pearson_sim()` above (only users who rated *both* restaurants count). Starting values reproduce the printed table exactly.

## Content-Based Filtering — Cosine Similarity on Item Features

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Item features: [avg_price_for_two, spice_level(1-5), is_vegetarian]
items = pd.DataFrame({
    'avg_price': [600, 450, 1200, 300],
    'spice_level': [4, 3, 4, 1],
    'is_veg': [1, 1, 0, 1],
}, index=ratings.columns)

sim_matrix = cosine_similarity(items)
sim_df = pd.DataFrame(sim_matrix, index=items.index, columns=items.index)
# If a user liked "Punjabi Tadka", recommend the most SIMILAR item by features (not behaviour)
most_similar = sim_df['Punjabi Tadka'].drop('Punjabi Tadka').idxmax()
print(f"Most feature-similar to Punjabi Tadka: {most_similar}")

## Evaluating Recommenders — Rating Accuracy Isn't the Real Question

The matrix factorization example above evaluated recommendations by how close the predicted rating was to the true rating (RMSE-style thinking). But no user ever sees "predicted 3.71 vs actual 3.5" — they see a **ranked list**. A model can have mediocre rating-prediction accuracy while still ranking the right items at the top, and vice versa. Production recommenders are judged on ranking quality, not rating-prediction error.

$$\mathrm{Precision@}k = \dfrac{\#\text{ relevant items in top } k \text{ recommended}}{k}$$

In [ ]:
import numpy as np

# For each user, rank ALL unrated restaurants by predicted rating, then check the top 2
def precision_at_k(predicted_ranking, relevant_items, k=2):
    top_k = predicted_ranking[:k]
    hits = len(set(top_k) & set(relevant_items))
    return hits / k

# Priya's true preferences (items she'd rate 4+): Barbeque Nation, Sagar Ratna
# Model's ranked recommendation list for Priya, best first:
priyas_ranking = ['Barbeque Nation', 'Cafe Coffee Day', 'Sagar Ratna']
priyas_relevant = ['Barbeque Nation', 'Sagar Ratna']

p_at_2 = precision_at_k(priyas_ranking, priyas_relevant, k=2)
print(f"Precision@2 for Priya: {p_at_2:.2f}")  # 1 of top 2 is truly relevant → 0.50

# Recall@k: of ALL relevant items, how many did the top-k list catch?
def recall_at_k(predicted_ranking, relevant_items, k=2):
    top_k = predicted_ranking[:k]
    hits = len(set(top_k) & set(relevant_items))
    return hits / len(relevant_items) if relevant_items else 0
print(f"Recall@2 for Priya: {recall_at_k(priyas_ranking, priyas_relevant, k=2):.2f}")

Unlike RMSE on raw ratings, precision@k and recall@k directly answer the business question: "of what we actually showed the user, how much was good — and of everything good, how much did we surface?" This is the same precision/recall trade-off from the classification chapters, just applied to a ranked list instead of a binary prediction.

| Metric | Answers | Blind spot |
|---|---|---|
| RMSE on predicted ratings | "How close were our numeric rating predictions?" | Doesn't measure whether the top-ranked items are actually the right ones |
| Precision@k | "Of what we showed, how much was relevant?" | Ignores ranking order within the top k |
| Recall@k | "Of everything relevant, how much did we surface?" | Can be gamed by making k very large |
| NDCG (Normalized Discounted Cumulative Gain) | "Are the BEST items ranked highest, not just present?" | More complex to compute and explain to stakeholders |

## Beyond Ranking Accuracy — System-Level Health Metrics

Precision@k, Recall@k, and NDCG all evaluate *one user's ranked list against their known preferences*. None of them catch a recommender that is technically accurate but practically broken — always recommending the same 10 bestsellers, feeling repetitive, or never surfacing anything from 80% of the catalogue. Production teams track a wider set of metrics for exactly this reason:

| Metric | Measures | Why it matters |
|---|---|---|
| Hit Rate | % of users for whom at least one held-out relevant item appears anywhere in their top-N list | A simple, intuitive user-level success/failure signal — easier to explain to stakeholders than Precision@k |
| Coverage | % of the entire item catalogue the system is capable of recommending at all | A model that only ever recommends the same bestsellers has low coverage — bad for surfacing Flipkart's long-tail inventory |
| Diversity | How different the items *within a single list* are from each other | Stops a top-5 list from being 5 near-identical items (e.g. five North Indian restaurants back-to-back) |
| Novelty | How non-obvious/unexpected the recommendations are — roughly the inverse of item popularity | Recommending only what's already trending everywhere adds little value over just showing a bestseller list |
| Churn | How much the recommendation list changes between visits, absent any new interaction | Too much churn feels random and untrustworthy; too little feels stale and ignores context |
| Responsiveness | How quickly a fresh interaction (clicking one product) visibly changes subsequent recommendations | Users expect the system to visibly react to what they just did, not lag behind by days |
| A/B Testing | Randomised live comparison of two recommender variants on real behaviour (click-through, purchase, revenue) | The only metric that directly measures business impact — every metric above is an offline proxy for this |

> **⚠ Offline Metrics Are Proxies, Not the Final Answer**
>
> A model can score well on Precision@k and NDCG while still under-performing in a live A/B test — for instance, by lacking diversity, feeling repetitive, or over-fitting to historical popularity in ways that don't translate to genuine engagement. Production teams treat offline metrics as a fast, cheap filter for iterating quickly, and reserve A/B testing as the final validation gate before a full rollout — exactly the same logic as using cross-validation before a real-world pilot elsewhere in this course.

## The Cold-Start Problem

| Scenario | Problem | Fix |
|---|---|---|
| New user, no ratings yet | Collaborative filtering has no behaviour to work from | Ask onboarding preference questions; show popularity-based defaults; use content-based filtering on stated preferences |
| New item, no ratings yet | Matrix factorization can't place it in latent space | Content-based filtering using item metadata until enough interactions accumulate |
| Both new (marketplace launch) | No signal at all | Popularity/trending fallback, contextual bandits to explore efficiently |

Most production systems (Flipkart, Swiggy) use a **hybrid** approach — collaborative filtering as the primary signal once enough interaction history exists, falling back to content-based or popularity-based recommendations for new users/items. Trust matters too: a system that's easily gamed by fake ratings or bot clicks (malicious user behaviour) will eventually poison its own collaborative-filtering signal, which is one reason production platforms invest heavily in fraud/abuse detection — the anomaly-detection techniques covered earlier in this course — specifically around ratings and review data.

## Beyond Personalisation — Association Rule Mining

A completely different question about purchases ignores *who* the customer is and asks only which items appear *in the same basket* — the "frequently bought together" logic. That technique, **Association Rule Mining** (support, confidence, lift, and the Apriori / FP-Growth algorithms), now has its own dedicated chapter.

> **🔗 Continues on the next page**
>
> Association Rule Mining is covered in full on the page, including an interactive support/confidence/lift calculator and a visual demonstration of the Apriori pruning principle.

> **🔗 Real-World Link — Fashion Recommendations**
>
> A content-based fashion recommender — suggesting visually/stylistically similar items from product attributes — is one of the most common real applications of the similarity techniques on this page. [See the case study →](https://statso.io/fashion-recommendations-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Cosine similarity of two users

Two users rated the same 4 restaurants. Write `cosine(u, v)` = dot product / product of norms, and store their similarity in `sim`.

In [ ]:
import numpy as np
def cosine(u, v):
    pass   # TODO
u = np.array([5, 4, 1, 2.0])
v = np.array([4, 5, 1, 1.0])
sim = None


In [ ]:
try:
    check("identical vectors -> 1", abs(cosine(u, u) - 1) < 1e-12)
    check("value", round(sim, 4) == 0.9668)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))
u = np.array([5, 4, 1, 2.0])
v = np.array([4, 5, 1, 1.0])
sim = cosine(u, v)

```

</details>

### Exercise 2 · Medium · User-based prediction

Predict a user's rating for an unseen restaurant as the **similarity-weighted average** of the neighbours' ratings: `sum(sim_i * r_i) / sum(|sim_i|)`. Write `predict(sims, ratings)` and store the prediction in `pred`.

In [ ]:
import numpy as np
def predict(sims, ratings):
    pass   # TODO
sims = np.array([0.9, 0.5, 0.1])
neighbour_ratings = np.array([5.0, 3.0, 1.0])
pred = None


In [ ]:
try:
    check("weighted average", abs(pred - (0.9 * 5 + 0.5 * 3 + 0.1 * 1) / 1.5) < 1e-12)
    check("pulled towards the most similar user", pred > 3.0)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def predict(sims, ratings):
    return float(np.sum(sims * ratings) / np.sum(np.abs(sims)))
sims = np.array([0.9, 0.5, 0.1])
neighbour_ratings = np.array([5.0, 3.0, 1.0])
pred = predict(sims, neighbour_ratings)

```

</details>

### Exercise 3 · Stretch · Recall@k

Write `recall_at_k(ranked, relevant, k)`: the share of the user's relevant items that appear in the top `k` of the ranking. (The lesson's `precision_at_k` divides by `k`; recall divides by the number of relevant items.)

In [ ]:
def recall_at_k(ranked, relevant, k):
    pass   # TODO


In [ ]:
try:
    ranked = ["a", "b", "c", "d", "e"]
    check("two of three relevant items in top 3", abs(recall_at_k(ranked, ["a", "c", "e"], 3) - 2 / 3) < 1e-12)
    check("none found", recall_at_k(ranked, ["x", "y"], 5) == 0)
    check("all found", recall_at_k(ranked, ["a", "b"], 2) == 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

```

</details>

---
*Back to the course: **Machine Learning End To End → Recommender Systems**.*